# Modular subtraction

Adapted from Qualtran v0.7.0, Copyright 2024 Google LLC, Apache-2.0.
Negation and subtraction preserve the source register, control polarity and arithmetic ordering.

In [ ]:
from collections import Counter
from dataclasses import replace
from IPython.display import SVG, display
from examples.algorithms.field_addition_catalog import field_signature_svg
from examples.algorithms.modular_subtraction_catalog import modular_subtraction_examples
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.dynamic import DynamicCircuit

examples = modular_subtraction_examples()

def show_signature(name):
    display(SVG(field_signature_svg([(name, examples[name].signature, {})])))

def show_calls(example):
    graph = example.call_graph()
    display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
    display(graph.leaf_counts)
    print("Source estimate (CCZ equivalents):", example.source_ccz_count())

## `ModNeg`
Performs modular negation.

Applies the operation $\ket{x} \rightarrow \ket{-x \% p}$

Note: This implements the decomposition from Fig 6 in but doesn't match table 8
since we don't use measurement based uncompution because that introduces random phase flips.

#### Parameters
 - `dtype`: Datatype of the register.
 - `p`: The modulus for the negation. 

#### Registers
 - `x`: The register contraining the integer we negate. 

#### References
 - [How to compute a 256-bit elliptic curve private key with only 50 million Toffoli gates](https://arxiv.org/abs/2306.08585). Fig 6b


In [ ]:
assert "mod_neg" in examples

### Example Instances

In [ ]:
mod_neg = examples["mod_neg"]

#### Graphical Signature

In [ ]:
show_signature("mod_neg")

### Call Graph

In [ ]:
show_calls(mod_neg)

## `CModNeg`
Performs controlled modular negation.

Applies the operation $\ket{c}\ket{x} \rightarrow \ket{c}\ket{(-1)^c x\%p}$

Note: while this matches the count from Fig 8, it's a different decomposition that controls
only the Add operation instead of turning the CNOTs into toffolis.

#### Parameters
 - `dtype`: Datatype of the register.
 - `p`: The modulus for the negation.
 - `cv`: value at which the gate is active. 

#### Registers
 - `ctrl`: Control bit.
 - `x`: The register contraining the integer we negate. 

#### References
 - [How to compute a 256-bit elliptic curve private key with only 50 million Toffoli gates](https://arxiv.org/abs/2306.08585). Fig 6b and 8


In [ ]:
assert "cmod_neg" in examples

### Example Instances

In [ ]:
cmod_neg = examples["cmod_neg"]

#### Graphical Signature

In [ ]:
show_signature("cmod_neg")

### Call Graph

In [ ]:
show_calls(cmod_neg)

## `ModSub`
Performs modular subtraction.

Applies the operation $\ket{x} \ket{y} \rightarrow \ket{x} \ket{y-x \% p}$

#### Parameters
 - `dtype`: Datatype of the register.
 - `p`: The modulus for the negation. 

#### Registers
 - `x`: The register contraining the first integer.
 - `y`: The register contraining the second integer. 

#### References
 - [How to compute a 256-bit elliptic curve private key with only 50 million Toffoli gates](https://arxiv.org/abs/2306.08585). Fig 6c and 8


In [ ]:
assert "modsub_symb" in examples

### Example Instances

In [ ]:
modsub_symb = examples["modsub_symb"]

#### Graphical Signature

In [ ]:
show_signature("modsub_symb")

### Call Graph

In [ ]:
show_calls(modsub_symb)

## `CModSub`
Performs controlled modular subtraction.

Applies the operation $\ket{c}\ket{x} \ket{y} \rightarrow \ket{c}\ket{x} \ket{y-cx \% p}$

#### Parameters
 - `dtype`: Datatype of the register.
 - `p`: The modulus for the negation.
 - `cv`: value at which the bloq is active. 

#### Registers
 - `ctrl`: control register.
 - `x`: The register contraining the first integer.
 - `y`: The register contraining the second integer. 

#### References
 - [How to compute a 256-bit elliptic curve private key with only 50 million Toffoli gates](https://arxiv.org/abs/2306.08585). Fig 6c and 8


In [ ]:
assert "cmodsub_symb" in examples

### Example Instances

In [ ]:
cmodsub_symb = examples["cmodsub_symb"]

#### Graphical Signature

In [ ]:
show_signature("cmodsub_symb")

### Call Graph

In [ ]:
show_calls(cmodsub_symb)

### Append the concrete circuits

Use big-endian residue words in the interval `[0, p)`. The locked constant load requires `p + 1` to fit in the word. The original low-level negation can leave a dirty flag outside the residue domain. The concrete append examples below report emitted instructions separately from the source models. Each example also appends its exact component adjoint.

In [ ]:
for example in examples.values():
    polarities = (0, 1) if example.kind.startswith("C") else (1,)
    for cv in polarities:
        op = replace(example, width=3, modulus=5, control_value=cv)
        circuit = DynamicCircuit(op.qubit_count)
        circuit.append(op)
        forward = circuit.to_ir().instructions
        counts = Counter(gate.name for gate in forward)
        print(op.kind, cv, "measured IR:", dict(counts), "qubits:", op.qubit_count)
        print("Source estimate (T equivalents):", 4 * op.source_ccz_count())
        circuit.append(replace(op, adjoint=True))
        assert len(circuit.to_ir().instructions) == 2 * len(forward)